<a href="https://colab.research.google.com/github/zty2020china/IOS13-SimulateTouch/blob/master/%E7%84%A6%E7%82%89%E7%83%AD%E5%B7%A5%E4%BB%BF%E7%9C%9F%E5%B9%B3%E5%8F%B0V1_6_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
焦炉专业封装软件 V1.6.1 - 饱和温湿联动 + 数据库可视化上传版
1. 解决 Mac 复杂的路径与权限隔离问题，新增前端拖拽上传 CSV 功能
2. 集成饱和水汽体积换算质量逻辑
"""

import os
import math
import gradio as gr
import pandas as pd
from pydantic import BaseModel

# Mac 环境网络补丁
os.environ["HTTP_PROXY"] = ""
os.environ["HTTPS_PROXY"] = ""
os.environ["NO_PROXY"] = "localhost,127.0.0.1,0.0.0.0"

WATER_VAPOR_DENSITY = 804.0  # 标况水蒸气密度 g/Nm3

def get_water_mass_by_temp(temp_c):
    """根据温度计算饱和水质量 (g/Nm3)"""
    vol_m3 = 0.0084 * math.exp(0.0558 * temp_c)
    return vol_m3 * WATER_VAPOR_DENSITY

# ==========================================
# 2. 数据库引擎 (增强上传能力)
# ==========================================
class OvenDatabase:
    def __init__(self):
        self.db = {}
        # 启动时依然尝试去同目录下找一下
        try:
            base_path = os.path.dirname(os.path.abspath(__file__))
            default_path = os.path.join(base_path, "焦炉常用参数 - 工作表1.csv")
            self.load_from_csv(default_path)
        except:
            pass

        if not self.db:
            self.load_demo_data()

    def load_demo_data(self):
        """如果没有数据，提供内置演示数据防崩溃"""
        self.db = {
            "等待上传CSV... (内置7.63m)": {"每孔炭化室装煤量（t）": "57.3", "周转时间": "25", "成焦率": "75", "炭化室长（mm）": "18560"},
            "等待上传CSV... (内置JN43)": {"每孔炭化室装煤量（t）": "17.9", "周转时间": "18", "成焦率": "73", "炭化室长（mm）": "14080"}
        }

    def load_from_csv(self, csv_path):
        if not csv_path or not os.path.exists(csv_path):
            return False

        for enc in ['gbk', 'utf-8-sig', 'gb2312', 'utf-8']:
            try:
                df = pd.read_csv(csv_path, encoding=enc, index_col=0)
                df_t = df.T

                new_db = {}
                for model_name, row_data in df_t.iterrows():
                    new_db[str(model_name).strip()] = {k.strip(): str(v) for k, v in row_data.items() if pd.notna(v)}

                if len(new_db) > 0:
                    self.db = new_db  # 成功解析才替换当前数据库
                    return True
            except:
                continue
        return False

    def get_models(self):
        return list(self.db.keys())

    def get_params(self, name):
        return self.db.get(name, {})

oven_db = OvenDatabase()

# ==========================================
# 3. 计算引擎 (保持专业逻辑)
# ==========================================
class CokeOvenEngine:
    @staticmethod
    def run_simulation(fuel_temp, fuel_hum, air_temp, air_rh, alpha, coal_mt, sched_count, sched_chg, sched_time, sched_rate):
        lhv = 17500.0 # kJ/Nm3 (COG典型值)
        v0_air = 4.25  # 理论空气量

        p_sat_air = 6.112 * math.exp((17.62 * air_temp) / (243.12 + air_temp))
        x_w_air = (air_rh / 100.0) * p_sat_air / 1013.25
        v_act_air = (alpha * v0_air) / (1 - x_w_air)

        v_h2o_fuel = fuel_hum / WATER_VAPOR_DENSITY
        v_act_flue = 5.0 + (alpha-1)*v0_air + v_h2o_fuel + (v_act_air * x_w_air)

        dry_coal = sched_chg * (1 - coal_mt/100.0)
        annual_cap = (sched_count / sched_time) * 8000 * sched_chg * (sched_rate/100.0)

        q_dry = 2350 + (coal_mt * 25)
        hourly_dry_kg = (sched_count / sched_time) * dry_coal * 1000.0
        total_flue_h = (hourly_dry_kg * q_dry / lhv) * v_act_flue

        return {
            "capacity": annual_cap,
            "metrics": {
                "燃料入炉含水质量": f"{fuel_hum:.2f} g/Nm³ (据{fuel_temp}℃饱和点计算)",
                "综合干煤耗热量": f"{q_dry:.0f} kJ/kg",
                "全炉实际总废气量": f"{total_flue_h:.0f} Nm³/h",
                "单标准燃烧室废气量": f"**{total_flue_h / sched_count:.0f} Nm³/h**"
            }
        }

# ==========================================
# 4. 前端界面与交互
# ==========================================
def on_temp_change(temp, auto_mode):
    if auto_mode:
        return round(get_water_mass_by_temp(temp), 2)
    return gr.update()

def on_model_change(name):
    if not name: return 30, 20, 75, "-"
    d = oven_db.get_params(name)
    info = f"**物理尺寸**: 长{d.get('炭化室长（mm）','-')}mm | 高{d.get('炭化室高（mm）','-')}mm"
    return float(d.get("每孔炭化室装煤量（t）", 30)), float(d.get("周转时间", 20)), float(d.get("成焦率", 75)), info

def handle_csv_upload(filepath):
    """处理用户在网页端上传的 CSV"""
    if not filepath:
        return gr.update()
    success = oven_db.load_from_csv(filepath)
    if success:
        models = oven_db.get_models()
        return gr.update(choices=models, value=models[0])
    else:
        return gr.update(label="❌ 格式错误，请检查CSV")

def main_calc(f_temp, f_hum, a_temp, a_rh, alpha, c_mt, o_cnt, o_chg, o_time, o_rate):
    res = CokeOvenEngine.run_simulation(f_temp, f_hum, a_temp, a_rh, alpha, c_mt, o_cnt, o_chg, o_time, o_rate)

    report = f"### 📊 焦炉热工集成计算书 (V1.6.1)\n\n"
    report += f"**项目单年标定产能：{res['capacity']/10000:.2f} 万吨/年** (8000h工制)\n\n"
    report += "| 参数名称 | 计算结果 |\n| :--- | :--- |\n"
    for k, v in res["metrics"].items():
        report += f"| {k} | {v} |\n"
    return report

with gr.Blocks(title="焦炉综合仿真 V1.6.1") as app:
    gr.Markdown("# 🏭 焦炉热工仿真平台 V1.6.1 (带数据库上传模块)")

    with gr.Row():
        with gr.Column():
            with gr.Group():
                gr.Markdown("### 📚 炉型特征数据库")
                # 新增：文件上传控件
                csv_upload = gr.File(label="📂 找不到型号？点这里直接上传 CSV 文件", file_types=[".csv"], type="filepath")

                model_dd = gr.Dropdown(choices=oven_db.get_models(), label="选择炉型", value=oven_db.get_models()[0])
                model_info = gr.Markdown("-")

                with gr.Row():
                    o_cnt = gr.Number(65, label="焦炉孔数 N")
                    o_chg = gr.Number(label="装煤量(t)")
                with gr.Row():
                    o_time = gr.Number(label="周转时间(h)")
                    o_rate = gr.Number(label="成焦率(%)")

            with gr.Group():
                gr.Markdown("### 💧 燃料温湿联动引擎")
                f_temp = gr.Slider(20, 45, value=35, label="燃料煤气温度 (℃)")
                auto_mode = gr.Checkbox(True, label="自动同步饱和含水量")
                f_hum = gr.Number(47.2, label="煤气含湿量 (g/Nm³)")

        with gr.Column():
            with gr.Group():
                gr.Markdown("### ⚙️ 工况与煤质设置")
                alpha = gr.Slider(1.1, 1.4, 1.2, label="空气过剩系数")
                c_mt = gr.Slider(5, 15, 10, label="入炉煤水分 %")
                a_temp = gr.Slider(0, 40, 25, label="环境气温 (℃)")
                a_rh = gr.Slider(0, 100, 60, label="空气相对湿度 %")

            btn = gr.Button("🚀 执行测算", variant="primary", size="lg")
            res_md = gr.Markdown("### 📑 计算报告\n等待执行...")

    # 事件绑定
    csv_upload.upload(handle_csv_upload, inputs=[csv_upload], outputs=[model_dd])
    model_dd.change(on_model_change, model_dd, [o_chg, o_time, o_rate, model_info])
    f_temp.change(on_temp_change, [f_temp, auto_mode], f_hum)
    btn.click(main_calc, [f_temp, f_hum, a_temp, a_rh, alpha, c_mt, o_cnt, o_chg, o_time, o_rate], res_md)
    app.load(on_model_change, model_dd, [o_chg, o_time, o_rate, model_info])

if __name__ == "__main__":
    app.launch(server_name="127.0.0.1", server_port=7860)